# Toxic Content Detection

You're an ML engineer at a social media platform. Millions of comments arrive every day, and the platform needs to detect toxic content at scale. But "toxic" is itself a contested concept. Sarcasm, cultural references, and context all matter.

Who labelled this data? Would everyone agree with the labels? A comment that reads as banter in one community might be genuinely harmful in another. Keep that in the back of your mind as you work through this lab.

In [ ]:
%pip install seaborn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay)

## Loading and exploring the data

In [ ]:
df = pd.read_csv("./comments.csv")
print(f"Shape: {df.shape}")
print(f"Class distribution:\n{df['is_toxic'].value_counts()}")
print(f"\nToxic rate: {df['is_toxic'].mean():.1%}")

Before we do anything with numbers, let's actually read some of the comments. This is especially important for text data. You need a feel for what "toxic" and "not toxic" look like in practice, and whether the labels seem reasonable.

In [ ]:
print("=== Non-toxic examples ===")
for text in df[df["is_toxic"] == 0].sample(5, random_state=42)["comment_text"]:
    print(f"  {text}")

print("\n=== Toxic examples ===")
for text in df[df["is_toxic"] == 1].sample(5, random_state=42)["comment_text"]:
    print(f"  {text}")

Look at these examples. Some are clear-cut. Others are ambiguous. "Maybe stick to topics you actually understand": is that toxic or just blunt disagreement? These edge cases are where content moderation gets difficult.

## Text to numbers

Machine learning models work with numbers, not words. We need to convert text into a numerical representation. The simplest approach: count how often each word appears.

### Bag of words

The simplest approach is to count how often each word appears in a piece of text. Let's see how this works with a small example before applying it to our full dataset.

In [ ]:
# Simple example first
example_texts = ["I love this video", "This video is terrible", "I love terrible videos"]
count_vec = CountVectorizer()
counts = count_vec.fit_transform(example_texts)
pd.DataFrame(counts.toarray(), columns=count_vec.get_feature_names_out(), index=example_texts)

Each text becomes a row of word counts. This is called "bag of words". It captures which words appear but throws away word order. "I love terrible videos" and "terrible videos I love" look identical.

### TF-IDF

TF-IDF (Term Frequency-Inverse Document Frequency) improves on raw counts. Words that appear everywhere (like "the", "is") get downweighted. Words that are distinctive to a particular comment get upweighted.

We'll use TF-IDF for our actual model. Before we can transform the text, we need to split the data into training and test sets, just as we did with numeric data in previous labs. The vectorizer learns its vocabulary from the training set only, to avoid data leakage.

In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["comment_text"], df["is_toxic"], test_size=0.2, random_state=42, stratify=df["is_toxic"]
)

tfidf = TfidfVectorizer(max_features=5000, stop_words="english")
X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

print(f"Vocabulary size: {len(tfidf.vocabulary_)}")
print(f"Feature matrix shape: {X_train_tfidf.shape}")
print(f"\nSample feature names: {tfidf.get_feature_names_out()[:20].tolist()}")

## Building a text classifier

Now that we have numbers instead of words, we can train a model. We'll use logistic regression again, the same algorithm from the previous two labs. The difference is that our features are now word frequencies rather than physical measurements.

Let's train it and see how accuracy looks as a first check.

In [ ]:
# Train a logistic regression on the TF-IDF features
model = LogisticRegression(max_iter=1000, random_state=42)
# Fit the model on training data
model.fit(X_train_tfidf, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test_tfidf)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")

## The confusion matrix

You know precision (unit 4) and recall (unit 3). Now let's see the full picture. The confusion matrix shows all four outcomes:

|  | Predicted: Not toxic | Predicted: Toxic |
|--|---------------------|------------------|
| **Actual: Not toxic** | True Negative (TN) | False Positive (FP), over-moderation |
| **Actual: Toxic** | False Negative (FN), missed harm | True Positive (TP) |

In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Not toxic", "Toxic"])
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap="Blues")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

Read the matrix. How many toxic comments did the model miss (bottom-left)? How many innocent comments did it flag (top-right)?

## Introducing F1

Over-moderation silences users and drives them away (false positives). Missing toxic content exposes users to harm (false negatives). Neither precision nor recall alone captures this tension. F1 is the harmonic mean of precision and recall, and it's high only when both are high.

In [ ]:
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1 Score:  {f1:.3f}")

F1 = 2 × (precision × recall) / (precision + recall). If either precision or recall is 0, F1 is 0. It penalises models that sacrifice one for the other.

## Class weighting for text

Just as we did in the fraud detection lab, we can use `class_weight="balanced"` to tell the model that errors on the minority class (toxic comments) should count for more. Let's see whether this shifts the precision-recall balance and improves F1.

In [ ]:
# Train a new model with class_weight="balanced"
model_balanced = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
model_balanced.fit(X_train_tfidf, y_train)
y_pred_balanced = model_balanced.predict(X_test_tfidf)

print(f"Precision: {precision_score(y_test, y_pred_balanced):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred_balanced):.3f}")
print(f"F1 Score:  {f1_score(y_test, y_pred_balanced):.3f}")

Compare these numbers to the unweighted model above. Did class weighting improve F1? Look at what happened to precision and recall individually. Class weighting often boosts recall (catching more toxic content) at the cost of some precision (more false alarms). Whether that's a good trade-off depends on what the platform values more: catching harm, or avoiding over-moderation.

## Error analysis

Numbers only tell part of the story. To really understand where a text classifier struggles, we need to look at the actual comments it got wrong. This is **error analysis**, and for NLP problems, it's often more revealing than any metric.

In [ ]:
# Find misclassified examples
test_df = pd.DataFrame({
    "text": X_test_text.values,
    "actual": y_test.values,
    "predicted": y_pred_balanced
})
errors = test_df[test_df["actual"] != test_df["predicted"]]

label_map = {0: "Not toxic", 1: "Toxic"}

print(f"Total errors: {len(errors)}\n")
print("=== False positives (flagged but not toxic) ===")
fps = errors[errors["predicted"] == 1]
for _, row in fps.head(5).iterrows():
    print(f"  \"{row['text']}\"")

print("\n=== False negatives (missed toxic content) ===")
fns = errors[errors["predicted"] == 0]
for _, row in fns.head(5).iterrows():
    print(f"  \"{row['text']}\"")

Look at the errors. Why did the model fail on these?

- **False positives:** did the comment use words that are often toxic but weren't here? (e.g. "sick" meaning "great")
- **False negatives:** was the toxicity expressed through sarcasm, innuendo, or context rather than specific words?

This is the fundamental limitation of bag-of-words: it sees words, not meaning. "This is sick" (positive slang) and "You make me sick" (negative) contain the same word but mean entirely different things.

## The transformer comparison (optional)

Modern NLP has moved beyond word counting. Transformer models like BERT read words in context. They understand that "sick" means something different in "this is sick" vs "you make me sick". Let's load a pretrained toxic content classifier and see if it handles the edge cases better.

**Note:** This section downloads a neural network (~50MB) and may take 30-60 seconds. It's optional. Skip it if your browser struggles.

In [ ]:
import micropip
await micropip.install("transformers_js_py")
from transformers_js_py import import_transformers_js
transformers = await import_transformers_js()
classifier = await transformers.pipeline("text-classification", "Xenova/toxic-bert")
print("Transformer model loaded!")

In [ ]:
# Run the transformer on the same misclassified examples
print("=== Transformer predictions on our model's errors ===\n")

for _, row in errors.head(10).iterrows():
    result = await classifier(row["text"])
    transformer_label = "Toxic" if result[0]["label"] == "toxic" else "Not toxic"
    our_label = label_map[row["predicted"]]
    actual_label = label_map[row["actual"]]
    print(f"Text: \"{row['text']}\"")
    print(f"  Actual: {actual_label} | Our model: {our_label} | Transformer: {transformer_label}")
    print()

Does the transformer get these right? In many cases, yes, because it understands context, word order, and sarcasm in ways that TF-IDF cannot. This is why the industry moved to transformer-based models. But they're also much larger, slower, and more expensive to run at scale.